# Day 19 Revision Summary — Week 4 Capstone

- **The five-step workflow is the real lesson.** Data → engineer a feature → one `ColumnTransformer` pipeline → bake off a few models with cross-validation → explain the winner. Almost every tabular ML task is a variation of this same shape.
- **There is no "best model" in the abstract.** On Day 16's cancer data, Logistic Regression won because the signal was smooth and linear (0.981 vs 0.958). On today's delivery data, Random Forest won because the signal had non-linear kinks — rush hours, and bikes on long trips (0.791 vs 0.784). That's why you always run the bake-off instead of guessing.
- **`ColumnTransformer` preps mixed columns in one object.** `StandardScaler` for numeric columns, `OneHotEncoder(handle_unknown="ignore")` for text columns, wrapped in a `Pipeline` with the model — one object, no leakage, works with `cross_val_score` out of the box.
- **Feature importance turns a score into a story.** `pipe.named_steps["clf"].feature_importances_`, paired with `pipe.named_steps["prep"].get_feature_names_out()`, tells you *why* the winner decided what it did (distance and prep time dominated the late-delivery call) — something you can hand to a manager, not just a number.
- **k-means gives you a second, label-free lens on the same data.** Scale first, pick `k` with the silhouette score, then read what each cluster's averages look like. Today's silhouettes were low (~0.25–0.28) — a reminder that "the clusters exist" and "the clusters are sharp" are different claims.


## Today's material (Day 19)

Three sources combine into today's work:

- **Classwork** (`classwork/data_gen.py`, `classwork/capstone.py`, `classwork/explore_clusters.py`) — the in-class exercises, run live during the two-hour session.
- **Homework** (`homework/homework.md`) — the take-home self-study, due before Week 5.
- **The slide deck's closing homework slide** (`slides/week4_day4_revision_capstone.html`) — restates the same homework (Titanic capstone + your own feature + commit) and previews Week 5 (neural networks).

All of it is broken out below, in order: classwork first, then homework.


## Classwork Exercise 1 — Run the capstone (`data_gen.py` + `capstone.py`)

**What's being asked:** Build the delivery dataset with `make_deliveries()` and check the late share (~0.52). Build the `ColumnTransformer` (scale `distance`/`prep`/`hour`/`is_rush`, one-hot `weather`/`vehicle`) and the three-model bake-off (Logistic, Decision Tree depth 4, Random Forest 300 trees). Cross-validate all three with `cv=5`, print `mean ± std`, and name the winner. **Stretch:** drop the engineered `is_rush` column and re-run — does every model get a little worse?

**Approach:**
1. `classwork/data_gen.py` and `classwork/capstone.py` already contain the full, runnable reference solution — start by running `python capstone.py` as-is and reading the printed mean/std for each model.
2. Confirm the winner matches the slide's result: Random Forest at 0.791 ± 0.015, edging out Logistic (0.784) and the depth-4 tree (0.772).
3. For the stretch goal, copy the numeric column list and drop `"is_rush"` from it, rebuild the same `ColumnTransformer` + three pipelines, and re-run `cross_val_score` for each.
4. Compare each new mean to its original — if `is_rush` was carrying real signal, every model's score should dip slightly.


In [ ]:
from data_gen import make_deliveries
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

df, late = make_deliveries()

# TODO 1: Same ColumnTransformer as capstone.py, but WITHOUT "is_rush".
# numeric_no_rush = ["distance", "prep", "hour"]
# categorical = ["weather", "vehicle"]
# pre = ColumnTransformer([
#     ("num", StandardScaler(), numeric_no_rush),
#     ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
# ])

# TODO 2: Rebuild the same three pipelines around this smaller `pre`.
# models = {
#     "Logistic (scaled)": Pipeline([("prep", pre), ("clf", LogisticRegression(max_iter=1000))]),
#     "Decision tree (d=4)": Pipeline([("prep", pre), ("clf", DecisionTreeClassifier(max_depth=4, random_state=42))]),
#     "Random forest (300)": Pipeline([("prep", pre), ("clf", RandomForestClassifier(n_estimators=300, random_state=42))]),
# }

# TODO 3: Cross-validate each and compare to the with-is_rush numbers
# (0.784 / 0.772 / 0.791) — did every model dip a little?
# for name, model in models.items():
#     s = cross_val_score(model, df, late, cv=5)
#     print(f"{name:22} {s.mean():.3f} +/- {s.std():.3f}")


## Classwork Exercise 2 & 3 — Explain & explore (feature importances + `explore_clusters.py`)

**What's being asked:** Fit the winning Random Forest pipeline and print its `feature_importances_` — do they match the story (distance and prep time dominate)? Run `explore_clusters.py`, read the silhouette scores for k=2..5, and describe the k=3 segments. Write two sentences on what you'd tell the ops team to reduce late orders. **Stretch:** add one-hot `weather` and `vehicle` to the clustering features — do the segments change?

**Approach:**
1. Fit the same Random Forest pipeline from Exercise 1 on the full data (`pipe.fit(df, late)`).
2. Pull readable column names with `pipe.named_steps["prep"].get_feature_names_out()` and the scores with `pipe.named_steps["clf"].feature_importances_`, then `sorted(zip(names, importances), key=lambda p: p[1], reverse=True)`.
3. Confirm against the slide's numbers: distance 0.351, prep 0.248, hour 0.123, weather_sunny 0.088, is_rush 0.054, weather_snow 0.038.
4. Run `classwork/explore_clusters.py` as-is, note the silhouette per k (weak but real: k=2 .245, k=3 .250, k=4 .281, k=5 .276), and read the k=3 segment averages (near/afternoon, mid-range/morning, far/evening-rush).
5. Write two sentences connecting the two analyses — e.g. long-distance evening-rush orders are both the likeliest to be late *and* their own cluster, so that's the segment ops should target first.
6. For the stretch, one-hot encode `weather`/`vehicle` (e.g. `pd.get_dummies`), concatenate onto the scaled numeric columns used for clustering, and re-run k-means to see if the segment boundaries shift.


In [ ]:
from data_gen import make_deliveries
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

df, late = make_deliveries()
numeric = ["distance", "prep", "hour", "is_rush"]
categorical = ["weather", "vehicle"]
pre = ColumnTransformer([
    ("num", StandardScaler(), numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
])
pipe = Pipeline([("prep", pre), ("clf", RandomForestClassifier(n_estimators=300, random_state=42))])

# TODO 1: Fit on the full data and read off the ranked feature importances.
# pipe.fit(df, late)
# names = pipe.named_steps["prep"].get_feature_names_out()
# importances = pipe.named_steps["clf"].feature_importances_
# ranked = sorted(zip(names, importances), key=lambda p: p[1], reverse=True)
# for name, score in ranked:
#     print(f"{score:.3f} {name}")

# TODO 2 (stretch): one-hot weather/vehicle and append to the clustering
# features from explore_clusters.py, then re-run KMeans + silhouette_score
# to see whether the k=3 segments change.


## Homework 1 — Capstone on real data (Titanic)

**What's being asked:** Load `sns.load_dataset("titanic")` — a real mixed dataset with numbers, categories, *and* missing values. Predict `survived`. Build a `ColumnTransformer` that imputes + scales the numeric columns and imputes + one-hot encodes the categorical columns, then bake off Logistic / Tree / Forest with `cross_val_score`. Who wins?

**Approach:**
1. `import seaborn as sns; df = sns.load_dataset("titanic")` and inspect `df.dtypes` / `df.isna().sum()` to see which columns are numeric, which are text, and where the gaps are (`age`, `embarked`, `deck` all have missing values).
2. Pick a feature set, e.g. numeric `["age", "fare", "sibsp", "parch"]` and categorical `["sex", "embarked", "pclass"]`.
3. Build the `ColumnTransformer` with imputation added in front of each half: `Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])` for numeric, `Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))])` for categorical.
4. Wrap each of Logistic Regression, a depth-limited Decision Tree, and a Random Forest around the same `pre`, exactly like `capstone.py`.
5. `cross_val_score(model, df[features], df["survived"], cv=5)` for each, print `mean ± std`, and name the winner.


In [ ]:
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# TODO 1: Load Titanic and pick the feature columns.
# titanic = sns.load_dataset("titanic")
# numeric = ["age", "fare", "sibsp", "parch"]
# categorical = ["sex", "embarked", "pclass"]
# X = titanic[numeric + categorical]
# y = titanic["survived"]

# TODO 2: ColumnTransformer with imputation in front of each half.
# num_pipe = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
# cat_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
#                      ("onehot", OneHotEncoder(handle_unknown="ignore"))])
# pre = ColumnTransformer([("num", num_pipe, numeric), ("cat", cat_pipe, categorical)])

# TODO 3: Same three-model bake-off shape as capstone.py.
# models = {
#     "Logistic": Pipeline([("prep", pre), ("clf", LogisticRegression(max_iter=1000))]),
#     "Tree (d=4)": Pipeline([("prep", pre), ("clf", DecisionTreeClassifier(max_depth=4, random_state=42))]),
#     "Forest (300)": Pipeline([("prep", pre), ("clf", RandomForestClassifier(n_estimators=300, random_state=42))]),
# }
# for name, model in models.items():
#     s = cross_val_score(model, X, y, cv=5)
#     print(f"{name:12} {s.mean():.3f} +/- {s.std():.3f}")


## Homework 2 — Engineer your own feature

**What's being asked:** Create one new column you believe matters — e.g. `family_size = sibsp + parch + 1`. Prove it helps by comparing the Homework 1 bake-off with and without it.

**Approach:**
1. Add the new column to a copy of the Titanic frame: `titanic["family_size"] = titanic["sibsp"] + titanic["parch"] + 1`.
2. Re-run the exact same `ColumnTransformer` + bake-off from Homework 1 twice: once on the original numeric list, once with `"family_size"` appended to it.
3. Compare each model's mean `cross_val_score` between the two runs.
4. Write one sentence on whether — and why — `family_size` moved the score (traveling alone vs. with a large family was a real survival factor on the Titanic, so it's a reasonable bet it helps the linear model most, matching Day 18's ratio/difference feature lesson).


In [ ]:
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# TODO 1: Titanic + the engineered family_size column.
# titanic = sns.load_dataset("titanic")
# titanic["family_size"] = titanic["sibsp"] + titanic["parch"] + 1

# TODO 2: Build the same pipeline shape as Homework 1, once WITHOUT and
# once WITH "family_size" in the numeric column list, and compare.
# numeric_base = ["age", "fare", "sibsp", "parch"]
# numeric_engineered = numeric_base + ["family_size"]
# categorical = ["sex", "embarked", "pclass"]
#
# def bake_off_score(numeric_cols):
#     num_pipe = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
#     cat_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
#                          ("onehot", OneHotEncoder(handle_unknown="ignore"))])
#     pre = ColumnTransformer([("num", num_pipe, numeric_cols), ("cat", cat_pipe, categorical)])
#     model = Pipeline([("prep", pre), ("clf", LogisticRegression(max_iter=1000))])
#     X = titanic[numeric_cols + categorical]
#     return cross_val_score(model, X, titanic["survived"], cv=5).mean()
#
# print("without family_size:", bake_off_score(numeric_base))
# print("with family_size   :", bake_off_score(numeric_engineered))


## Homework 3 — Explain it

**What's being asked:** Fit the winning model from Homework 1 (with the Homework 2 feature included). Print `feature_importances_` (Forest/Tree) or the coefficients (Logistic). Write two sentences on what drove survival.

**Approach:**
1. Refit the Homework 1 winner on the full feature set, including `family_size`.
2. If the winner is the Forest or the Tree: `pipe.named_steps["prep"].get_feature_names_out()` paired with `pipe.named_steps["clf"].feature_importances_`, ranked highest first (same recipe as the Classwork Exercise 2 & 3 cell above).
3. If the winner is Logistic Regression: use `pipe.named_steps["clf"].coef_[0]` instead — larger magnitude (either sign) means a bigger swing in predicted survival; the sign tells you the direction.
4. Write two sentences naming the top 2-3 features and what they suggest (e.g. `sex`, `pclass`, and `fare` typically dominate Titanic survival — women, first class, and higher fares correlate with getting to a lifeboat).


In [ ]:
# TODO 1: Refit the Homework 1/2 winning pipeline (numeric_engineered + categorical, full data).
# pipe.fit(X, y)

# TODO 2a: Tree/Forest winner — ranked feature importances.
# names = pipe.named_steps["prep"].get_feature_names_out()
# importances = pipe.named_steps["clf"].feature_importances_
# ranked = sorted(zip(names, importances), key=lambda p: p[1], reverse=True)
# for name, score in ranked:
#     print(f"{score:.3f} {name}")

# TODO 2b: Logistic winner — coefficients instead.
# names = pipe.named_steps["prep"].get_feature_names_out()
# coefs = pipe.named_steps["clf"].coef_[0]
# ranked = sorted(zip(names, coefs), key=lambda p: abs(p[1]), reverse=True)
# for name, coef in ranked:
#     print(f"{coef:+.3f} {name}")

# TODO 3: Two sentences on what drove survival, naming the top features.


## Homework 4 — Predict then run

**What's being asked:** Before running the cell below, predict whether growing the delivery-lateness Random Forest from 300 to 1000 trees will do much better. The forest scored 0.791 with 300 trees.

**Prediction (write yours before running):** More trees mostly reduces the *variance* of a Random Forest's predictions by averaging over more of them — it does not add new signal the existing trees couldn't already see, since they're all bootstrap-sampling the same features. So the expected direction is a flat-to-tiny improvement, with most of the gain already captured by 300 trees (the classic "more trees stops helping past a point" lesson from Day 16).

**Approach:**
1. Rebuild the exact `capstone.py` `ColumnTransformer` and feature list.
2. Loop `n_estimators` over `[50, 300, 1000]`, rebuilding the Random Forest pipeline each time.
3. Run `cross_val_score(pipe, df, late, cv=5).mean()` for each and print `n, score`.
4. Confirm (or correct) your prediction against the actual numbers.


In [ ]:
from data_gen import make_deliveries
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

df, late = make_deliveries()
pre = ColumnTransformer([
    ("num", StandardScaler(), ["distance", "prep", "hour", "is_rush"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["weather", "vehicle"]),
])
for n in [50, 300, 1000]:
    pipe = Pipeline([("prep", pre), ("clf", RandomForestClassifier(n_estimators=n, random_state=42))])
    print(n, round(cross_val_score(pipe, df, late, cv=5).mean(), 3))
# Lesson: more trees stops helping past a point (Day 16).


## Coming up — Week 5

**Neural networks** — Week 4's classical-ML toolkit (trees, forests, k-means, feature engineering, pipelines, bake-offs) is done. Week 5 leaves classical ML behind and builds the foundation of modern AI: neurons, layers, weights, and how a network learns by adjusting them.
